<a href="https://colab.research.google.com/github/tayyba6/ML_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline rule

I will prioritize pages using a simple directional score based on their March search-performance signals. Pages with stronger search exposure and weaker search performance receive higher priority for review.

The baseline uses **March impressions, CTR, and average position**. The score is intended for **prioritization and decision-support**, not as proof that a page needs a specific content change.

### Reason codes

* **HIGH_EXPOSURE** — the page has relatively high March impressions, so changes could affect a meaningful amount of search exposure.
* **LOW_CTR_OPPORTUNITY** — the page has relatively low CTR compared with other pages, making it worth reviewing, but this does not prove that a CTR fix will improve clicks.
* **WEAK_POSITION** — the page has a relatively poor March average search position, indicating weaker search visibility.
* **MULTI_SIGNAL** — the page shows more than one of the above conditions and therefore receives stronger review priority.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

I convert the three confirmed March signals into percentile-based directional scores. Higher impressions increase priority because the page has more search exposure. Lower CTR increases priority because it represents a possible click opportunity. Worse average position increases priority because it represents weaker search visibility.

The three components are combined with equal weight. This is a baseline prioritization rule, not a predictive model. The score is used to decide which pages should be reviewed first.

In [8]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_Token")

print("HF token loaded successfully:", bool(HF_TOKEN))

HF token loaded successfully: True


In [9]:
# ML-07 — Section 2: Build the ranked queue

import duckdb
import numpy as np
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# 1. Load Hugging Face access token
# ---------------------------------------------------------

from google.colab import userdata

HF_TOKEN = userdata.get("HF_Token")

if not HF_TOKEN:
    raise ValueError("HF_Token was not found in Colab Secrets.")

# ---------------------------------------------------------
# 2. Connect to the FlyRank warehouse
# ---------------------------------------------------------

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
    """
)

FACT = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

# ---------------------------------------------------------
# 3. Build the March feature frame
# ---------------------------------------------------------
# Only March data is used for the baseline score.
# April outcome data is NOT used here.

feature_sql = f"""
WITH march AS (
    SELECT *
    FROM read_parquet('{FACT}')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
)

SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS gsc_impressions_march,

    SUM(gsc_clicks) AS gsc_clicks_march,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
        ELSE NULL
    END AS gsc_ctr_march,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions)
        ELSE NULL
    END AS gsc_avg_position_march,

    SUM(
        CASE
            WHEN ga4_data_available IS TRUE
            THEN ga4_sessions
            ELSE NULL
        END
    ) AS ga4_sessions_march

FROM march

GROUP BY
    client_hash_id,
    content_hash_id
"""

features = con.sql(feature_sql).df()

print("Feature frame shape:", features.shape)
display(features.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (176738, 7)


,client_hash_id,content_hash_id,gsc_impressions_march,gsc_clicks_march,gsc_ctr_march,gsc_avg_position_march,ga4_sessions_march
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,0.000000,4.311688,NaN
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10849.0,22.0,0.002028,8.049866,NaN
2,client_62f4a7e64f5e0096,content_e689bc511192751a,61.0,0.0,0.000000,5.885246,NaN
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,705.0,1.0,0.001418,5.863830,NaN
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0,0.0,0.000000,14.360000,NaN


### Scoring method

I rank each page on three March signals using percentile ranks.

- Higher impressions receive a higher exposure score.
- Lower CTR receives a higher opportunity score.
- Worse average position receives a higher visibility-risk score.

The three directional scores are given equal weight and averaged into one baseline action score.

This score is only used to prioritize pages for review. It does not predict future clicks or prove that a page needs a specific content change.

In [10]:
# ML-07 — Section 2: Create directional scores and ranked queue

ranked = features.copy()

# Remove rows where the confirmed GSC signals needed for scoring are missing.
ranked = ranked.dropna(
    subset=[
        "gsc_impressions_march",
        "gsc_ctr_march",
        "gsc_avg_position_march"
    ]
).copy()

# ---------------------------------------------------------
# Percentile-based directional scores
# ---------------------------------------------------------

# Higher impressions = higher priority
ranked["exposure_score"] = (
    ranked["gsc_impressions_march"]
    .rank(method="average", pct=True)
)

# Lower CTR = higher priority
ranked["low_ctr_score"] = 1 - (
    ranked["gsc_ctr_march"]
    .rank(method="average", pct=True)
)

# Worse average position = higher priority
ranked["weak_position_score"] = (
    ranked["gsc_avg_position_march"]
    .rank(method="average", pct=True)
)

# ---------------------------------------------------------
# Equal-weight baseline action score
# ---------------------------------------------------------

ranked["baseline_action_score"] = (
    ranked["exposure_score"]
    + ranked["low_ctr_score"]
    + ranked["weak_position_score"]
) / 3

# ---------------------------------------------------------
# Reason-code flags
# ---------------------------------------------------------

ranked["HIGH_EXPOSURE"] = (
    ranked["exposure_score"] >= 0.75
)

ranked["LOW_CTR_OPPORTUNITY"] = (
    ranked["low_ctr_score"] >= 0.75
)

ranked["WEAK_POSITION"] = (
    ranked["weak_position_score"] >= 0.75
)

# ---------------------------------------------------------
# Reason code
# ---------------------------------------------------------

def make_reason_code(row):
    reasons = []

    if row["HIGH_EXPOSURE"]:
        reasons.append("HIGH_EXPOSURE")

    if row["LOW_CTR_OPPORTUNITY"]:
        reasons.append("LOW_CTR_OPPORTUNITY")

    if row["WEAK_POSITION"]:
        reasons.append("WEAK_POSITION")

    if len(reasons) >= 2:
        return "MULTI_SIGNAL"

    if len(reasons) == 1:
        return reasons[0]

    return "BASELINE_PRIORITY"

ranked["reason_code"] = ranked.apply(make_reason_code, axis=1)

# ---------------------------------------------------------
# Rank everything
# ---------------------------------------------------------

ranked = ranked.sort_values(
    "baseline_action_score",
    ascending=False
).reset_index(drop=True)

ranked["baseline_rank"] = np.arange(1, len(ranked) + 1)

# ---------------------------------------------------------
# Write required CSV
# ---------------------------------------------------------

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

ranked[
    [
        "baseline_rank",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions_march",
        "gsc_ctr_march",
        "gsc_avg_position_march",
        "exposure_score",
        "low_ctr_score",
        "weak_position_score",
        "baseline_action_score",
        "reason_code"
    ]
].to_csv(output_path, index=False)

print("Ranked queue shape:", ranked.shape)
print("CSV written to:", output_path)

display(
    ranked[
        [
            "baseline_rank",
            "client_hash_id",
            "content_hash_id",
            "gsc_impressions_march",
            "gsc_ctr_march",
            "gsc_avg_position_march",
            "baseline_action_score",
            "reason_code"
        ]
    ].head(20)
)

Ranked queue shape: (176738, 16)
CSV written to: work/outputs/baseline_action_score.csv


,baseline_rank,client_hash_id,content_hash_id,gsc_impressions_march,gsc_ctr_march,gsc_avg_position_march,baseline_action_score,reason_code
0,1,client_3197e6291363b4db,content_65b8a4998e633d89,8905.0,0.0,76.919708,0.880680,MULTI_SIGNAL
1,2,client_23a62021009f63c4,content_295e883e0e86ca3c,21939.0,0.0,61.379142,0.880547,MULTI_SIGNAL
2,3,client_23a62021009f63c4,content_bbf8e4d669f253cf,28934.0,0.0,47.853667,0.870718,MULTI_SIGNAL
3,4,client_e547b89c05043229,content_4002467a580a7f98,11973.0,0.0,54.937443,0.870386,MULTI_SIGNAL
4,5,client_23a62021009f63c4,content_959d535a9fcc865c,10452.0,0.0,53.172407,0.867099,MULTI_SIGNAL
5,6,client_23a62021009f63c4,content_066bb7aeff9aeea8,11443.0,0.0,50.999913,0.866395,MULTI_SIGNAL
6,7,client_23a62021009f63c4,content_2da022341f8803c3,15285.0,0.0,47.299836,0.866148,MULTI_SIGNAL
7,8,client_23a62021009f63c4,content_e8700175bf54e3d5,19586.0,0.0,45.030379,0.865779,MULTI_SIGNAL
8,9,client_e547b89c05043229,content_6177aad2ded9dee5,10333.0,0.0,50.557050,0.864616,MULTI_SIGNAL
9,10,client_23a62021009f63c4,content_421ad263ab8c8c03,10190.0,0.0,50.367223,0.864288,MULTI_SIGNAL


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top 20 pages are review priorities, not confirmed problems. For each page I record the main reason code, a confidence note about the strength of the prioritization signal, and what could make the recommendation wrong.

The review is intentionally cautious because aggregate March signals do not show query intent, SERP features, seasonality, or the actual quality of the page content.

In [11]:
# ML-07 — Section 3: Top-20 review

top20 = ranked.head(20).copy()

def confidence_note(row):
    signal_count = sum([
        row["HIGH_EXPOSURE"],
        row["LOW_CTR_OPPORTUNITY"],
        row["WEAK_POSITION"]
    ])

    if signal_count == 3:
        return "Higher review priority because all three signals agree; still directional, not causal."
    elif signal_count == 2:
        return "Moderate review confidence because two signals agree; outcome is not guaranteed."
    elif signal_count == 1:
        return "Lower review confidence because one signal mainly drives the priority."
    else:
        return "Low confidence; score is driven by relative ranking without a strong reason-code flag."

def what_would_make_it_wrong(row):
    reasons = []

    if row["HIGH_EXPOSURE"]:
        reasons.append(
            "High impressions may come from queries where the page is already appropriate."
        )

    if row["LOW_CTR_OPPORTUNITY"]:
        reasons.append(
            "Low CTR may be explained by SERP features, query intent, or the type of search."
        )

    if row["WEAK_POSITION"]:
        reasons.append(
            "Average position is an aggregate and may hide differences across queries."
        )

    if not reasons:
        reasons.append(
            "The relative score may not reflect a real content opportunity."
        )

    return " ".join(reasons)

top20["action"] = (
    "Review page content, search intent, and SERP presentation before making changes."
)

top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    what_would_make_it_wrong,
    axis=1
)

top20_review = top20[
    [
        "baseline_rank",
        "client_hash_id",
        "content_hash_id",
        "baseline_action_score",
        "reason_code",
        "action",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
].copy()

display(top20_review)

,baseline_rank,client_hash_id,content_hash_id,baseline_action_score,reason_code,action,confidence_note,what_would_make_it_wrong
0,1,client_3197e6291363b4db,content_65b8a4998e633d89,0.880680,MULTI_SIGNAL,"Review page content, search intent, and SERP p...",Moderate review confidence because two signals...,High impressions may come from queries where t...
1,2,client_23a62021009f63c4,content_295e883e0e86ca3c,0.880547,MULTI_SIGNAL,"Review page content, search intent, and SERP p...",Moderate review confidence because two signals...,High impressions may come from queries where t...
2,3,client_23a62021009f63c4,content_bbf8e4d669f253cf,0.870718,MULTI_SIGNAL,"Review page content, search intent, and SERP p...",Moderate review confidence because two signals...,High impressions may come from queries where t...
3,4,client_e547b89c05043229,content_4002467a580a7f98,0.870386,MULTI_SIGNAL,"Review page content, search intent, and SERP p...",Moderate review confidence because two signals...,High impressions may come from queries where t...
4,5,client_23a62021009f63c4,content_959d535a9fcc865c,0.867099,MULTI_SIGNAL,"Review page content, search intent, and SERP p...",Moderate review confidence because two signals...,High impressions may come from queries where t...
5,6,client_23a62021009f63c4,content_066bb7aeff9aeea8,0.866395,MULTI_SIGNAL,"Review page content, search intent, and SERP p...",Moderate review confidence because two signals...,High impressions may come from queries where t...
6,7,client_23a62021009f63c4,content_2da022341f8803c3,0.866148,MULTI_SIGNAL,"Review page content, search intent, and SERP p...",Moderate review confidence because two signals...,High impressions may come from queries where t...
7,8,client_23a62021009f63c4,content_e8700175bf54e3d5,0.865779,MULTI_SIGNAL,"Review page content, search intent, and SERP p...",Moderate review confidence because two signals...,High impressions may come from queries where t...
8,9,client_e547b89c05043229,content_6177aad2ded9dee5,0.864616,MULTI_SIGNAL,"Review page content, search intent, and SERP p...",Moderate review confidence because two signals...,High impressions may come from queries where t...
9,10,client_23a62021009f63c4,content_421ad263ab8c8c03,0.864288,MULTI_SIGNAL,"Review page content, search intent, and SERP p...",Moderate review confidence because two signals...,High impressions may come from queries where t...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some top-ranked pages may be weak picks because the score can be driven mainly by one directional signal. In particular, zero-CTR pages can receive high priority even when the page's actual search situation requires more context.

I therefore inspect top-20 pages with only one strong reason code rather than treating every high-ranked page as a confirmed opportunity.

The leakage check confirms that the baseline uses March feature data only. April outcome data is not used to calculate the score.

In [12]:
# ML-07 — Section 4: Weak picks

# Pages in the Top 20 where only one of the three signals
# crosses the strong-signal threshold.
top20["signal_count"] = (
    top20["HIGH_EXPOSURE"].astype(int)
    + top20["LOW_CTR_OPPORTUNITY"].astype(int)
    + top20["WEAK_POSITION"].astype(int)
)

weak_picks = top20[
    top20["signal_count"] <= 1
].copy()

print("Potential weak picks in Top 20:", len(weak_picks))

display(
    weak_picks[
        [
            "baseline_rank",
            "client_hash_id",
            "content_hash_id",
            "gsc_impressions_march",
            "gsc_ctr_march",
            "gsc_avg_position_march",
            "baseline_action_score",
            "reason_code"
        ]
    ]
)

Potential weak picks in Top 20: 0


,baseline_rank,client_hash_id,content_hash_id,gsc_impressions_march,gsc_ctr_march,gsc_avg_position_march,baseline_action_score,reason_code


In [13]:
# ---------------------------------------------------------
# Product/private-field check
# ---------------------------------------------------------

product_or_private_terms = [
    "product",
    "query",
    "url",
    "keyword",
    "page_url",
    "client_name"
]

suspicious_columns = [
    col for col in ranked.columns
    if any(term in col.lower() for term in product_or_private_terms)
]

print("Potential product/private fields in output:", suspicious_columns)

assert "client_name" not in ranked.columns
assert "page_url" not in ranked.columns

print("Output privacy check: PASS")

Potential product/private fields in output: []
Output privacy check: PASS


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.